# 📦 Module 10 — Modules, Packages & the Standard Library

## 🎯 What you will learn
1. What a module and a package are; all forms of `import`
2. How Python finds modules (`sys.path`), `__name__ == "__main__"`
3. Creating your own module and package
4. Virtual environments and `pip`
5. A guided tour of the most useful standard-library modules: `math`, `random`, `datetime`, `os`/`pathlib`, `sys`, `collections`, `itertools`, `functools`, `json`, `csv`, `re`, `time`, `statistics`, `string`, `textwrap`, `logging`, `argparse`, `heapq`, `bisect`, `decimal`, `fractions`, `enum`, `dataclasses`, `typing`
6. Hidden facts & tricks

**Exam relevance:** ⭐⭐⭐⭐ (module/package definitions, import forms, `random`, `datetime`, `math`)

---
## 1. Modules
A **module** is simply a `.py` file containing definitions (functions, classes, variables). A **package** is a folder of modules (with an `__init__.py`). A **library** is a collection of packages.

Python's motto: **"batteries included"** — the standard library has 200+ modules.

### Ways to import

In [1]:
import math                          # 1. whole module; use math.sqrt(...)
print(math.sqrt(16))

from math import sqrt, pi            # 2. specific names; use directly
print(sqrt(25), pi)

import math as m                     # 3. alias
print(m.floor(2.7))

from math import factorial as fact   # 4. alias for a name
print(fact(5))

from math import *                   # 5. everything (❌ avoid: pollutes namespace)
print(ceil(2.1))

4.0
5.0 3.141592653589793
2
120
3


### What happens on `import`?
1. Python checks `sys.modules` (cache). If found, reuse it.
2. Otherwise searches the folders in `sys.path`: current dir → `PYTHONPATH` → standard library → `site-packages`.
3. Runs the module's code **once** and stores the module object in `sys.modules`.

In [2]:
import sys
print(type(sys.path), len(sys.path) > 0)
print("math" in sys.modules)
print(math.__name__, math.__doc__.splitlines()[0])
print([n for n in dir(math) if not n.startswith("_")][:8], "...")

<class 'list'> True
True
math This module provides access to the mathematical functions
['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt'] ...


### Making your own module
Let's create a file on the fly and import it (normally you'd just save `mytools.py` next to your script).

In [3]:
import importlib, pathlib, sys
pathlib.Path("mytools.py").write_text('''
"""My tiny helper module."""
PI = 3.14159

def area(r):
    return PI * r * r

class Greeter:
    def hello(self): return "hello from Greeter"

if __name__ == "__main__":
    print("running mytools directly")
''')
sys.path.insert(0, ".")
import mytools
print(mytools.area(2), mytools.Greeter().hello(), mytools.__name__)

12.56636 hello from Greeter mytools


### `if __name__ == "__main__":` — the most important idiom
- When a file is **run directly**, `__name__` is `"__main__"`.
- When it is **imported**, `__name__` is the module's name.
So code under this `if` runs only when executed as a script, not when imported.

```python
def main():
    ...

if __name__ == "__main__":
    main()
```

### Packages

```
mypackage/
├── __init__.py        # marks the folder as a package (can be empty)
├── maths.py
└── text/
    ├── __init__.py
    └── clean.py
```
`import mypackage.maths`, `from mypackage.text import clean`, `from . import sibling` (relative import).

In [4]:
import os, pathlib
pkg = pathlib.Path("shapes"); pkg.mkdir(exist_ok=True)
(pkg / "__init__.py").write_text("from .circle import area as circle_area\n")
(pkg / "circle.py").write_text("import math\ndef area(r): return math.pi * r * r\n")
import shapes
print(round(shapes.circle_area(1), 4))
from shapes import circle
print(circle.__name__, shapes.__file__.endswith("__init__.py"))

3.1416
shapes.circle True


## 2. Installing third-party packages: `pip` & virtual environments

```bash
pip install requests            # install
pip install numpy==2.0.1        # exact version
pip uninstall requests
pip list                        # what's installed
pip freeze > requirements.txt   # save exact versions
pip install -r requirements.txt # recreate the environment
```

**Virtual environment** = an isolated Python + packages per project (prevents version clashes):

```bash
python -m venv .venv
.venv\Scripts\activate          # Windows
source .venv/bin/activate       # macOS / Linux
deactivate
```

Popular packages: `numpy`, `pandas`, `matplotlib`, `requests`, `flask`/`fastapi`, `django`, `scikit-learn`, `pytest`, `beautifulsoup4`, `pillow`.

## 3. Standard library tour
Run each cell and read the comments.

### 3.1 `math` (recap) and `statistics`

In [5]:
import math, statistics as st
data = [2, 4, 4, 4, 5, 5, 7, 9]
print(st.mean(data), st.median(data), st.mode(data))
print(round(st.stdev(data), 3), st.variance(data), st.quantiles(data, n=4))
print(math.dist((0, 0), (3, 4)), math.degrees(math.pi), math.prod([1, 2, 3, 4]))

5 4.5 4
2.138 4.571428571428571 [4.0, 4.5, 6.5]
5.0 180.0 24


### 3.2 `random` — pseudo-random numbers

In [6]:
import random
random.seed(42)                         # reproducible results
print(random.random())                  # float in [0, 1)
print(random.randint(1, 6))             # int inclusive both ends
print(random.randrange(0, 10, 2))       # like range()
print(random.uniform(1.5, 2.5))
items = ["a", "b", "c", "d", "e"]
print(random.choice(items))
print(random.choices(items, k=3))       # with replacement
print(random.sample(items, 3))          # without replacement
random.shuffle(items); print(items)     # in place
print(random.gauss(0, 1))

# Secure randomness (passwords, tokens) -> use `secrets`, not random
import secrets
print(len(secrets.token_hex(8)), secrets.choice("abc") in "abc")

0.6394267984578837
1
4
1.7448918538034763
b
['d', 'd', 'e']
['a', 'd', 'e']
['b', 'c', 'd', 'e', 'a']
-0.2317928608380212
16 True


### 3.3 `datetime` & `time`

In [7]:
from datetime import datetime, date, timedelta, time as dtime
d = date(2025, 3, 14)
print(d, d.year, d.month, d.day, d.weekday(), d.strftime("%A, %d %B %Y"))
dt = datetime(2025, 3, 14, 15, 30, 45)
print(dt, dt.strftime("%d/%m/%Y %I:%M %p"), dt.isoformat())
print(d + timedelta(days=30), dt - timedelta(hours=5))
print((date(2025, 12, 25) - d).days, "days to Christmas from Pi day")
parsed = datetime.strptime("2024-02-29 08:15", "%Y-%m-%d %H:%M")
print(parsed, parsed.date().isocalendar())
print(type(datetime.now()), date.today().year >= 2025)

import time
t0 = time.perf_counter()
time.sleep(0.01)
print(f"slept ~{time.perf_counter() - t0:.2f}s", time.time() > 1.7e9)

2025-03-14 2025 3 14 4 Friday, 14 March 2025
2025-03-14 15:30:45 14/03/2025 03:30 PM 2025-03-14T15:30:45
2025-04-13 2025-03-14 10:30:45
286 days to Christmas from Pi day
2024-02-29 08:15:00 datetime.IsoCalendarDate(year=2024, week=9, weekday=4)
<class 'datetime.datetime'> True
slept ~0.01s True


Common `strftime` codes: `%Y` year, `%m` month, `%d` day, `%H` 24h, `%I` 12h, `%M` minute, `%S` second, `%p` AM/PM, `%A` weekday name, `%B` month name, `%j` day of year.

### 3.4 `os`, `pathlib`, `shutil`, `sys`

In [8]:
import os, sys
from pathlib import Path

print(os.getcwd() == str(Path.cwd()))
print(os.name, os.path.join("folder", "file.txt"))
print(os.path.splitext("report.final.pdf"), os.path.basename("/a/b/c.txt"), os.path.dirname("/a/b/c.txt"))
print(os.path.exists("nothing_here"), "PATH" in os.environ or "Path" in os.environ)

p = Path("demo_dir")
p.mkdir(exist_ok=True)
f = p / "note.txt"                        # / joins paths in pathlib!
f.write_text("hello pathlib")
print(f.read_text(), f.suffix, f.stem, f.parent, f.exists(), f.stat().st_size)
print(sorted(x.name for x in p.iterdir()), list(p.glob("*.txt")))

import shutil
shutil.copy(f, p / "copy.txt")
print(sorted(os.listdir(p)))
shutil.rmtree(p)

print(sys.argv[0] != "", sys.version_info.major, sys.maxsize)

True
nt folder\file.txt
('report.final', '.pdf') c.txt /a/b
False True
hello pathlib .txt note demo_dir True 13
['note.txt'] [WindowsPath('demo_dir/note.txt')]
['copy.txt', 'note.txt']
True 3 9223372036854775807


### 3.5 `collections`

In [9]:
from collections import Counter, defaultdict, deque, namedtuple, OrderedDict, ChainMap
print(Counter("abracadabra").most_common(3))
dd = defaultdict(lambda: "N/A"); print(dd["missing"])
dq = deque(maxlen=3)
for i in range(6): dq.append(i)
print(dq)                                       # keeps only the last 3
print(ChainMap({"a": 1}, {"a": 2, "b": 3})["b"])
od = OrderedDict.fromkeys("abc"); od.move_to_end("a"); print(list(od))

[('a', 5), ('b', 2), ('r', 2)]
N/A
deque([3, 4, 5], maxlen=3)
3
['b', 'c', 'a']


### 3.6 `json`, `csv` — data interchange

In [10]:
import json, csv, io
person = {"name": "Asha", "skills": ["py", "sql"], "age": 19, "active": True, "score": None}
s = json.dumps(person, indent=2)          # dict -> JSON string
print(s)
back = json.loads(s)                      # JSON string -> dict
print(back == person)
# json.dump(obj, file) / json.load(file) work with files

buf = io.StringIO()
w = csv.writer(buf)
w.writerow(["name", "marks"]); w.writerows([["Asha", 90], ["Ravi", 85]])
print(buf.getvalue())
buf.seek(0)
for row in csv.DictReader(buf):
    print(row)

{
  "name": "Asha",
  "skills": [
    "py",
    "sql"
  ],
  "age": 19,
  "active": true,
  "score": null
}
True
name,marks
Asha,90
Ravi,85

{'name': 'Asha', 'marks': '90'}
{'name': 'Ravi', 'marks': '85'}


### 3.7 `re` — regular expressions (mini preview; full module later)

In [11]:
import re
text = "Call 98765-43210 or 91234-56789 by 25/12/2025"
print(re.findall(r"\d{5}-\d{5}", text))
print(re.search(r"(\d{2})/(\d{2})/(\d{4})", text).groups())
print(re.sub(r"\d", "#", "a1b22"))

['98765-43210', '91234-56789']
('25', '12', '2025')
a#b##


### 3.8 `heapq`, `bisect` — priority queues and sorted lists

In [12]:
import heapq, bisect
h = [5, 1, 8, 3]
heapq.heapify(h)                            # min-heap in place
print(heapq.heappop(h), heapq.nsmallest(2, [9, 4, 7, 1]), heapq.nlargest(2, [9, 4, 7, 1]))
heapq.heappush(h, 0); print(h[0])           # h[0] is always the smallest
print(bisect.bisect([10, 20, 30], 25))

1 [1, 4] [9, 7]
0
2


### 3.9 `enum`, `dataclasses`, `typing`

In [13]:
from enum import Enum, auto
class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = auto()
print(Color.RED, Color.RED.name, Color.RED.value, Color(2), list(Color))

from dataclasses import dataclass, field
@dataclass(order=True)
class Student:
    name: str
    marks: int = 0
    tags: list = field(default_factory=list)
s = Student("Asha", 90)
print(s, s == Student("Asha", 90), Student("A", 1) < Student("B", 1))

from typing import Optional, Union
def find(x: int) -> Optional[str]:      # str or None
    return "y" if x else None
print(find(1), find(0))

Color.RED RED 1 Color.GREEN [<Color.RED: 1>, <Color.GREEN: 2>, <Color.BLUE: 3>]
Student(name='Asha', marks=90, tags=[]) True True
y None


### 3.10 `logging`, `argparse`

In [14]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("demo")
log.debug("hidden (level too low)")
log.info("app started")
log.warning("disk almost full")
log.error("something failed")

import argparse
parser = argparse.ArgumentParser(description="Greeter")
parser.add_argument("name")
parser.add_argument("--times", type=int, default=1)
args = parser.parse_args(["Asha", "--times", "2"])   # normally reads the command line
print(args.name * args.times)

AshaAsha


INFO: app started
ERROR: something failed


### 3.11 `sys`, `platform`, `timeit`, `hashlib`, `uuid`, `pickle`, `copy`, `pprint`

In [15]:
import hashlib, uuid, pickle, pprint, timeit, platform
print(hashlib.sha256(b"password").hexdigest()[:16])
print(len(str(uuid.uuid4())), platform.system() in ("Windows", "Linux", "Darwin"))
blob = pickle.dumps({"a": [1, 2, 3]}); print(pickle.loads(blob))
pprint.pprint({"z": [1, 2], "a": {"nested": True}}, width=30)
print(timeit.timeit("[i*i for i in range(100)]", number=1000) > 0)

5e884898da280471
36 True
{'a': [1, 2, 3]}
{'a': {'nested': True},
 'z': [1, 2]}
True


## 4. Reload & introspect a module

In [16]:
import importlib
importlib.reload(mytools)                # re-executes the file after you edit it
print(mytools.__file__.endswith("mytools.py"))
print([n for n in dir(mytools) if not n.startswith("__")])

True
['Greeter', 'PI', 'area']


---
# 🕵️ Hidden Facts & Pro Tricks

In [17]:
# 1. See where a module lives, and read its source
import inspect, textwrap
print(inspect.getsourcefile(textwrap).endswith("textwrap.py"))

# 2. Lazy import inside a function speeds up start-up
def heavy():
    import decimal
    return decimal.Decimal("1.1")
print(heavy())

# 3. importlib to import by string name
import importlib
mod = importlib.import_module("math")
print(mod.sqrt(81))

# 4. __all__ controls `from module import *`
# 5. Circular imports: two modules importing each other -> import inside functions to break the cycle
# 6. python -m module: run a module as a script (python -m json.tool file.json prettifies JSON)
# 7. `python -c "import antigravity"` opens xkcd; `python -m this` prints the Zen
# 8. Standard-library gems most people miss:
import functools, itertools, operator, statistics, fractions, graphlib, zoneinfo, tomllib, shlex, difflib
print(difflib.get_close_matches("pythn", ["python", "java", "cobol"]))
print(list(graphlib.TopologicalSorter({"app": {"lib", "utils"}, "lib": {"utils"}}).static_order()))
print(shlex.split("cp 'my file.txt' dest"))
print(tomllib.loads('name = "demo"\nversion = 3'))
from zoneinfo import ZoneInfo
from datetime import datetime
print(datetime(2025, 6, 1, 12, tzinfo=ZoneInfo("UTC")).astimezone(ZoneInfo("Asia/Kolkata")))

# 9. pathlib is the modern os.path: Path.home(), Path.glob("**/*.py"), Path.read_text()
# 10. `python -m pip install --upgrade pip` always uses the right Python
# 11. `pip install pipx` installs command-line apps in isolation; `uv` is a blazing-fast pip/venv alternative
# 12. shutil.get_terminal_size, shutil.which("python") locate programs
import shutil
print(shutil.which("python") is not None)

True
1.1
9.0
['python']
['utils', 'lib', 'app']
['cp', 'my file.txt', 'dest']
{'name': 'demo', 'version': 3}
2025-06-01 17:30:00+05:30
True


**Hidden fact:** importing a module *executes* it top to bottom exactly once. A module named the same as a stdlib module (e.g., your own `random.py`) will **shadow** it and cause baffling errors — never name files `random.py`, `math.py`, `test.py`, `string.py`.

**Hidden fact:** the standard library has a whole `sqlite3` database, an `http.server`, a `tkinter` GUI toolkit, `turtle` graphics, `unittest`, `smtplib`, `zipfile`, `urllib`, `threading`, `asyncio`, and `wave` — you can build a lot before touching pip.

---
# 📝 Exam Corner

**Theory**
1. What is a module? A package? What is `__init__.py`?
2. Different ways of importing — differences & when to use each.
3. What does `if __name__ == "__main__":` do?
4. What is `pip`? What is a virtual environment and why use it?
5. Explain `random` functions: `random()`, `randint()`, `choice()`, `shuffle()`, `sample()`.
6. Write code to print today's date in `dd-mm-yyyy`.
7. Difference between `import module` and `from module import name`.

**Quick code**

In [18]:
from datetime import date
print(date.today().strftime("%d-%m-%Y") != "")

import random
random.seed(7)
print(random.randint(1, 100), random.choice("abc"))

True
42 a


---
## ✅ Summary
- Module = `.py` file; package = folder with `__init__.py`; import executes once and is cached in `sys.modules`.
- Use `import x`, `from x import y`, `import x as z`; avoid `import *`.
- `__name__ == "__main__"` separates script use from import use.
- Use `venv` + `pip` per project. Know the stdlib: `math random datetime os pathlib sys collections itertools json csv re heapq`.

➡️ **Next:** Module 11 — Exceptions & Error Handling.